In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# NODE-SPECIFIC CONFIGURATION
# ============================================================

# ============================================================
# DIRECTORY CONFIGURATION
# ============================================================

ROOT = Path.cwd()

TRAINING_DATA_DIR = ROOT / "Data" / "training_data"

OUTPUT_DIR = ROOT / "Results" / "training_curves"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Current directory:", ROOT)
print("Training data directory:", TRAINING_DATA_DIR)
print("Output directory:", OUTPUT_DIR)


NODE_CONFIGS = {
    4: {
        "pnn_path": TRAINING_DATA_DIR / "4_PNN_model.json",
        "fnn_path": TRAINING_DATA_DIR / "4_FNN_model.json",
        "xticks": [0, 2000, 4000, 6000],
        "ylim": (1e-6, 1e0),
        "pnn_val_markers": 10,
        "fnn_val_markers": 10,
    },
    6: {
        "pnn_path": TRAINING_DATA_DIR / "6_PNN_model.json",
        "fnn_path": TRAINING_DATA_DIR / "6_FNN_model.json",
        "xticks": [0, 4000, 8000, 12000],
        "ylim": (1e-4, 1e0),
        "pnn_val_markers": 10,
        "fnn_val_markers": 10,
    },
    8: {
        "pnn_path": TRAINING_DATA_DIR / "8_PNN_model.json",
        "fnn_path": TRAINING_DATA_DIR / "8_FNN_model.json",
        "xticks": [0, 1000, 2000, 3000],
        "ylim": (1e-3, 1e0),
        "pnn_val_markers": 10,
        "fnn_val_markers": 4,
    },
}


# ============================================================
# GLOBAL SETTINGS
# ============================================================

OUTPUT_DIR = ROOT / "Results" / "training_curves"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_BASENAME = "pnn_fnn_training_dynamics_combined"

SAVE_DPI = 600
SHOW_PLOT = True

# Larger and wider figure
FIGSIZE = (25, 12)

# Extra x-axis space at beginning and end
X_MARGIN_FRACTION = 0.05

# Visual style
TRAIN_LINEWIDTH = 3
VAL_LINEWIDTH = 1.2
MARKER_SIZE = 4.8
MARKER_EDGEWIDTH = 1.3

SPINE_COLOR = "0.15"


# ============================================================
# COLOUR AND GRAYSCALE PALETTES
# ============================================================

PALETTES = {
    "colour": {
        "pnn": "#1F4E79",
        "fnn": "#B85C00",
    },
    "black_white_gray": {
        "pnn": "0.05",
        "fnn": "0.45",
    },
}


# ============================================================
# MATPLOTLIB STYLE
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],

    "axes.labelsize": 30,
    "axes.titlesize": 24,
    "xtick.labelsize": 23,
    "ytick.labelsize": 23,
    "legend.fontsize": 22,

    "axes.linewidth": 1.0,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4.0,
    "ytick.major.size": 4.0,

    "figure.dpi": 600,
    "savefig.dpi": SAVE_DPI,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# ============================================================
# DATA HELPERS
# ============================================================

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)


def get_array_from_possible_keys(data, possible_keys):
    for key in possible_keys:
        if key in data:
            return np.asarray(data[key], dtype=float)

    raise KeyError(
        "Could not find the required loss key.\n"
        f"Tried keys: {possible_keys}\n"
        f"Available keys: {list(data.keys())}"
    )


def extract_train_val_losses(json_path):
    data = load_json(json_path)

    train_keys = [
        "train_loss",
        "training_loss",
        "train_losses",
        "training_losses",
        "loss_train",
        "losses_train",
        "history_train_loss",
    ]

    val_keys = [
        "val_loss",
        "validation_loss",
        "val_losses",
        "validation_losses",
        "loss_val",
        "losses_val",
        "history_val_loss",
    ]

    train_loss = get_array_from_possible_keys(data, train_keys)
    val_loss = get_array_from_possible_keys(data, val_keys)

    return train_loss, val_loss


def make_epochs(loss_array):
    return np.arange(1, len(loss_array) + 1)


def choose_marker_indices(n_points, target_markers):
    if n_points <= target_markers:
        return np.arange(n_points)

    return np.unique(
        np.linspace(0, n_points - 1, target_markers, dtype=int)
    )


def marker_points(x, y, target_markers):
    idx = choose_marker_indices(len(x), target_markers)
    return x[idx], y[idx]


def compute_x_limits(pnn_epochs, fnn_epochs, xticks, margin_fraction=0.05):
    """
    Compute x-axis limits using both the tick range and actual curve lengths.

    This fixes the issue where curves can touch the right border when the
    final epoch extends beyond, or close to, the last x tick.
    """
    x_min = min(0, min(xticks), pnn_epochs[0], fnn_epochs[0])
    x_max = max(max(xticks), pnn_epochs[-1], fnn_epochs[-1])

    x_range = x_max - x_min

    x_left = x_min - margin_fraction * x_range
    x_right = x_max + margin_fraction * x_range

    return x_left, x_right


# ============================================================
# PLOT HELPERS
# ============================================================

def style_axis(ax):
    """Clean axis style with no grid."""

    ax.grid(False)

    ax.tick_params(
        axis="both",
        which="major",
        direction="out",
        width=1.0,
        length=4.0,
    )

    ax.tick_params(
        axis="y",
        which="minor",
        direction="out",
        width=0.7,
        length=2.5,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_color(SPINE_COLOR)


def add_node_title(ax, n):
    """Put node title above each subplot."""

    ax.set_title(
        f"{n}-node system",
        fontsize=24,
        pad=14,
    )


def add_best_validation_box(ax, pnn_val, fnn_val):
    """Put best validation-loss box in the top-right."""

    pnn_best = np.min(pnn_val)
    fnn_best = np.min(fnn_val)

    text = (
        "Best validation loss\n"
        f"PNN: {pnn_best:.2e}\n"
        f"FNN: {fnn_best:.2e}"
    )

    ax.text(
        0.97,
        0.97,
        text,
        transform=ax.transAxes,
        fontsize=18,
        linespacing=1.2,
        ha="right",
        va="top",
        bbox=dict(
            boxstyle="round,pad=0.32",
            facecolor="white",
            edgecolor="0.65",
            linewidth=0.8,
            alpha=0.95,
        ),
    )


def add_stop_epoch_annotation(ax, fnn_epochs, n, cfg, palette):
    """
    Mark where the 8-node FNN stopped.
    Arrow points to the x-axis intersection of the dotted line.
    """

    if n != 8:
        return

    fnn_color = palette["fnn"]
    stop_epoch = fnn_epochs[-1]

    ax.axvline(
        stop_epoch,
        color=fnn_color,
        linestyle=":",
        linewidth=1.4,
        alpha=0.95,
        zorder=1,
    )

    y_axis_intersection = cfg["ylim"][0]

    ax.annotate(
        f"FNN stopped at epoch {stop_epoch}",
        xy=(stop_epoch, y_axis_intersection),
        xytext=(stop_epoch + 150, cfg["ylim"][0] * 1.35),
        textcoords="data",
        fontsize=18,
        color=fnn_color,
        arrowprops=dict(
            arrowstyle="->",
            color=fnn_color,
            lw=1.0,
        ),
        bbox=dict(
            boxstyle="round,pad=0.22",
            facecolor="white",
            edgecolor=fnn_color,
            linewidth=0.8,
            alpha=0.95,
        ),
        ha="left",
        va="bottom",
    )


def plot_single_node(ax, n, cfg, palette):
    pnn_color = palette["pnn"]
    fnn_color = palette["fnn"]

    # -------------------------
    # Load data
    # -------------------------

    pnn_train, pnn_val = extract_train_val_losses(cfg["pnn_path"])
    fnn_train, fnn_val = extract_train_val_losses(cfg["fnn_path"])

    pnn_epochs = make_epochs(pnn_train)
    fnn_epochs = make_epochs(fnn_train)

    # -------------------------
    # Sparse validation markers
    # -------------------------

    pnn_marker_x, pnn_marker_y = marker_points(
        pnn_epochs,
        pnn_val,
        target_markers=cfg["pnn_val_markers"],
    )

    fnn_marker_x, fnn_marker_y = marker_points(
        fnn_epochs,
        fnn_val,
        target_markers=cfg["fnn_val_markers"],
    )

    # -------------------------
    # Training curves
    # -------------------------

    line_pnn_train, = ax.plot(
        pnn_epochs,
        pnn_train,
        color=pnn_color,
        linestyle="-",
        linewidth=TRAIN_LINEWIDTH,
        label="PNN training",
        zorder=3,
    )

    line_fnn_train, = ax.plot(
        fnn_epochs,
        fnn_train,
        color=fnn_color,
        linestyle="-",
        linewidth=TRAIN_LINEWIDTH,
        label="FNN training",
        zorder=3,
    )

    # -------------------------
    # Validation curves
    # -------------------------
    # These thick semi-transparent dashed curves are the "shadowy lines"
    # visible behind the training curves.
    # Kept unchanged because VAL_LINEWIDTH and alpha are preserved.

    ax.plot(
        pnn_epochs,
        pnn_val,
        color=pnn_color,
        linestyle="--",
        linewidth=VAL_LINEWIDTH,
        alpha=0.60,
        zorder=2,
    )

    ax.plot(
        fnn_epochs,
        fnn_val,
        color=fnn_color,
        linestyle="--",
        linewidth=VAL_LINEWIDTH,
        alpha=0.60,
        zorder=2,
    )

    mark_pnn_val, = ax.plot(
        pnn_marker_x,
        pnn_marker_y,
        color=pnn_color,
        marker="o",
        linestyle="None",
        markersize=MARKER_SIZE,
        markerfacecolor="white",
        markeredgecolor=pnn_color,
        markeredgewidth=MARKER_EDGEWIDTH,
        label="PNN validation",
        zorder=4,
    )

    mark_fnn_val, = ax.plot(
        fnn_marker_x,
        fnn_marker_y,
        color=fnn_color,
        marker="s",
        linestyle="None",
        markersize=MARKER_SIZE,
        markerfacecolor="white",
        markeredgecolor=fnn_color,
        markeredgewidth=MARKER_EDGEWIDTH,
        label="FNN validation",
        zorder=4,
    )

    # -------------------------
    # Axes
    # -------------------------

    ax.set_yscale("log")
    ax.set_ylim(cfg["ylim"])

    x_left, x_right = compute_x_limits(
        pnn_epochs=pnn_epochs,
        fnn_epochs=fnn_epochs,
        xticks=cfg["xticks"],
        margin_fraction=X_MARGIN_FRACTION,
    )

    ax.set_xlim(x_left, x_right)
    ax.set_xticks(cfg["xticks"])

    # -------------------------
    # Labels and annotations
    # -------------------------

    add_node_title(ax, n)
    add_best_validation_box(ax, pnn_val, fnn_val)
    add_stop_epoch_annotation(ax, fnn_epochs, n, cfg, palette)
    style_axis(ax)

    return line_pnn_train, line_fnn_train, mark_pnn_val, mark_fnn_val


def save_figure(fig, suffix):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    svg_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}_{suffix}.svg"
    pdf_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}_{suffix}.pdf"
    png_path = OUTPUT_DIR / f"{OUTPUT_BASENAME}_{suffix}.png"

    fig.savefig(svg_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")

    print(f"Saved SVG: {svg_path.resolve()}")
    print(f"Saved PDF: {pdf_path.resolve()}")
    print(f"Saved PNG: {png_path.resolve()}")

    if SHOW_PLOT:
        plt.show()

    plt.close(fig)


# ============================================================
# MAIN FIGURE FUNCTION
# ============================================================

def make_combined_figure(style_name, palette):
    fig, axes = plt.subplots(
        1,
        3,
        figsize=FIGSIZE,
        sharey=False,
    )

    legend_handles = None

    for ax, (n, cfg) in zip(axes, NODE_CONFIGS.items()):
        handles = plot_single_node(ax, n, cfg, palette)

        if legend_handles is None:
            legend_handles = handles

    # Only first subplot gets y-label
    axes[0].set_ylabel("Loss")
    axes[1].set_ylabel("")
    axes[2].set_ylabel("")

    # One shared x-label
    fig.supxlabel("Epoch", fontsize=30, y=0.055)

    # One common legend
    legend_labels = [
        "PNN training",
        "FNN training",
        "PNN validation",
        "FNN validation",
    ]

    fig.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=4,
        frameon=True,
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.75",
        handlelength=2.4,
        columnspacing=1.6,
        borderpad=0.35,
        bbox_to_anchor=(0.5, 0.98),
    )

    # Larger plotting region, with room for legend and titles
    fig.subplots_adjust(
        left=0.055,
        right=0.995,
        bottom=0.19,
        top=0.78,
        wspace=0.23,
    )

    save_figure(fig, suffix=style_name)


# ============================================================
# RUN
# ============================================================

make_combined_figure("colour", PALETTES["colour"])
make_combined_figure("black_white_gray", PALETTES["black_white_gray"])